In [3]:
!apt-get update -y
!apt-get install openjdk-11-jdk -y
!pip install pyspark

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,294 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 4,422 kB in 1s (3,375 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Don

In [4]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

In [8]:
import os
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
!java -version

JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [9]:
!pip install --upgrade pyspark

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("CMS Project") \
    .getOrCreate()

print("Spark started")
print(spark)

Spark started


In [2]:
# Load the beneficiary file
from google.colab import files
uploaded = files.upload()

beneficiarydf = spark.read.csv(
    "DE1_0_2008_Beneficiary_Summary_File_Sample_4.csv",
    header=True,
    inferSchema=True
)

Saving DE1_0_2008_Beneficiary_Summary_File_Sample_4.csv to DE1_0_2008_Beneficiary_Summary_File_Sample_4.csv


In [3]:
beneficiarydf.show(5)
beneficiarydf.printSchema()

+----------------+-------------+-------------+-----------------+------------+-------------+-------------+--------------+-----------------------+------------------------+------------------------+-----------------+-----------+------+-----------+-------+-------+-----------+-----------+-----------+-----------+--------+-----------+-----------+---------+---------+-----------+---------+---------+------------+----------+----------+
|     DESYNPUF_ID|BENE_BIRTH_DT|BENE_DEATH_DT|BENE_SEX_IDENT_CD|BENE_RACE_CD|BENE_ESRD_IND|SP_STATE_CODE|BENE_COUNTY_CD|BENE_HI_CVRAGE_TOT_MONS|BENE_SMI_CVRAGE_TOT_MONS|BENE_HMO_CVRAGE_TOT_MONS|PLAN_CVRG_MOS_NUM|SP_ALZHDMTA|SP_CHF|SP_CHRNKIDN|SP_CNCR|SP_COPD|SP_DEPRESSN|SP_DIABETES|SP_ISCHMCHT|SP_OSTEOPRS|SP_RA_OA|SP_STRKETIA|MEDREIMB_IP|BENRES_IP|PPPYMT_IP|MEDREIMB_OP|BENRES_OP|PPPYMT_OP|MEDREIMB_CAR|BENRES_CAR|PPPYMT_CAR|
+----------------+-------------+-------------+-----------------+------------+-------------+-------------+--------------+-----------------------+

In [4]:
beneficiarydf.columns
beneficiarydf.count()
beneficiarydf.select("DESYNPUF_ID").distinct().count()

116279

In [5]:
beneficiarydf.groupBy("BENE_SEX_IDENT_CD").count().show()
beneficiarydf.groupBy("SP_STATE_CODE").count().show()
beneficiarydf.groupBy("SP_ALZHDMTA").count().show()
beneficiarydf.filter(beneficiarydf["SP_DIABETES"] == 1).count()

+-----------------+-----+
|BENE_SEX_IDENT_CD|count|
+-----------------+-----+
|                1|51431|
|                2|64848|
+-----------------+-----+

+-------------+-----+
|SP_STATE_CODE|count|
+-------------+-----+
|           31| 3105|
|           53|  497|
|           34| 3880|
|           28|  692|
|           26| 2347|
|           27|  464|
|           44| 2729|
|           12|  589|
|           22| 2640|
|           47|  313|
|            1| 2593|
|           52| 2448|
|           13|  650|
|            6| 1971|
|           16| 1335|
|            3| 2261|
|           20|  713|
|           54| 1731|
|            5|10017|
|           19| 1710|
+-------------+-----+
only showing top 20 rows
+-----------+-----+
|SP_ALZHDMTA|count|
+-----------+-----+
|          1|22432|
|          2|93847|
+-----------+-----+



44077

In [6]:
# Load the inpatient file
from google.colab import files
uploaded = files.upload()

inpatientdf = spark.read.csv(
    "DE1_0_2008_to_2010_Inpatient_Claims_Sample_4.csv",
    header=True,
    inferSchema=True
)

Saving DE1_0_2008_to_2010_Inpatient_Claims_Sample_4.csv to DE1_0_2008_to_2010_Inpatient_Claims_Sample_4.csv


In [7]:
inpatientdf.columns

['DESYNPUF_ID',
 'CLM_ID',
 'SEGMENT',
 'CLM_FROM_DT',
 'CLM_THRU_DT',
 'PRVDR_NUM',
 'CLM_PMT_AMT',
 'NCH_PRMRY_PYR_CLM_PD_AMT',
 'AT_PHYSN_NPI',
 'OP_PHYSN_NPI',
 'OT_PHYSN_NPI',
 'CLM_ADMSN_DT',
 'ADMTNG_ICD9_DGNS_CD',
 'CLM_PASS_THRU_PER_DIEM_AMT',
 'NCH_BENE_IP_DDCTBL_AMT',
 'NCH_BENE_PTA_COINSRNC_LBLTY_AM',
 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM',
 'CLM_UTLZTN_DAY_CNT',
 'NCH_BENE_DSCHRG_DT',
 'CLM_DRG_CD',
 'ICD9_DGNS_CD_1',
 'ICD9_DGNS_CD_2',
 'ICD9_DGNS_CD_3',
 'ICD9_DGNS_CD_4',
 'ICD9_DGNS_CD_5',
 'ICD9_DGNS_CD_6',
 'ICD9_DGNS_CD_7',
 'ICD9_DGNS_CD_8',
 'ICD9_DGNS_CD_9',
 'ICD9_DGNS_CD_10',
 'ICD9_PRCDR_CD_1',
 'ICD9_PRCDR_CD_2',
 'ICD9_PRCDR_CD_3',
 'ICD9_PRCDR_CD_4',
 'ICD9_PRCDR_CD_5',
 'ICD9_PRCDR_CD_6',
 'HCPCS_CD_1',
 'HCPCS_CD_2',
 'HCPCS_CD_3',
 'HCPCS_CD_4',
 'HCPCS_CD_5',
 'HCPCS_CD_6',
 'HCPCS_CD_7',
 'HCPCS_CD_8',
 'HCPCS_CD_9',
 'HCPCS_CD_10',
 'HCPCS_CD_11',
 'HCPCS_CD_12',
 'HCPCS_CD_13',
 'HCPCS_CD_14',
 'HCPCS_CD_15',
 'HCPCS_CD_16',
 'HCPCS_CD_17',
 'HCPCS_CD_18',

In [8]:
inpatientdf.describe().show()

+-------+----------------+--------------------+--------------------+-------------------+--------------------+------------------+-----------------+------------------------+--------------------+-------------------+--------------------+--------------------+-------------------+--------------------------+----------------------+------------------------------+------------------------------+------------------+-------------------+-----------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----------+----------+----------+----------+----------+----------+----------+----------+----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------

In [9]:
inpatientdf.select("DESYNPUF_ID").distinct().count()

37715

### **SILVER LAYER** - Making data usable and trustworthy

In [10]:
# TypeCasting
#1. Converting BENE_BIRTH_DT, BENE_DEATH_DT = these columns to DATE  19380901
from pyspark.sql.functions import col, to_date

beneficiarydf = beneficiarydf.withColumn(
    "BENE_BIRTH_DT",
    to_date(col("BENE_BIRTH_DT").cast("string"), "yyyyMMdd")
)

beneficiarydf = beneficiarydf.withColumn(
    "BENE_DEATH_DT",
    to_date(col("BENE_DEATH_DT").cast("string"), "yyyyMMdd")
)

In [11]:
# Renaming columns from beneficiary df
rename_map = {
    "DESYNPUF_ID": "patient_id",
    "BENE_BIRTH_DT": "birth_date",
    "BENE_DEATH_DT": "death_date",
    "BENE_SEX_IDENT_CD": "gender_code",
    "BENE_RACE_CD": "race_code",
    "BENE_ESRD_IND": "esrd_indicator",
    "SP_STATE_CODE": "state_code",
    "BENE_COUNTY_CD": "county_code",
    "BENE_HI_CVRAGE_TOT_MONS": "hi_coverage_months",
    "BENE_SMI_CVRAGE_TOT_MONS": "smi_coverage_months",
    "BENE_HMO_CVRAGE_TOT_MONS": "hmo_coverage_months",
    "PLAN_CVRG_MOS_NUM": "plan_coverage_months",
    "SP_ALZHDMTA": "alzheimers",
    "SP_CHF": "congestive_heart_failure",
    "SP_CHRNKIDN": "chronic_kidney_disease",
    "SP_CNCR": "cancer",
    "SP_COPD": "copd",
    "SP_DEPRESSN": "depression",
    "SP_DIABETES": "diabetes",
    "SP_ISCHMCHT": "ischemic_heart_disease",
    "SP_OSTEOPRS": "osteoporosis",
    "SP_RA_OA": "rheumatoid_or_osteoarthritis",
    "SP_STRKETIA": "stroke_or_tia",
    "MEDREIMB_IP": "medicare_reimbursement_ip",
    "BENRES_IP": "beneficiary_responsibility_ip",
    "PPPYMT_IP": "primary_payer_payment_ip",
    "MEDREIMB_OP": "medicare_reimbursement_op",
    "BENRES_OP": "beneficiary_responsibility_op",
    "PPPYMT_OP": "primary_payer_payment_op",
    "MEDREIMB_CAR": "medicare_reimbursement_car",
    "BENRES_CAR": "beneficiary_responsibility_car",
    "PPPYMT_CAR": "primary_payer_payment_car"
}

for old_name, new_name in rename_map.items():
    beneficiarydf = beneficiarydf.withColumnRenamed(old_name, new_name)

In [12]:
# Renaming columns from inpatient
rename_map = {
    "DESYNPUF_ID": "patient_id",
    "CLM_ID": "claim_id",
    "SEGMENT": "claim_segment",
    "CLM_FROM_DT": "claim_start_date",
    "CLM_THRU_DT": "claim_end_date",
    "PRVDR_NUM": "provider_id",
    "CLM_PMT_AMT": "claim_payment_amount",
    "NCH_PRMRY_PYR_CLM_PD_AMT": "primary_payer_paid_amount",
    "AT_PHYSN_NPI": "attending_physician_npi",
    "OP_PHYSN_NPI": "operating_physician_npi",
    "OT_PHYSN_NPI": "other_physician_npi",
    "CLM_ADMSN_DT": "admission_date",
    "ADMTNG_ICD9_DGNS_CD": "admitting_diagnosis_code",
    "CLM_PASS_THRU_PER_DIEM_AMT": "pass_through_per_diem_amount",
    "NCH_BENE_IP_DDCTBL_AMT": "inpatient_deductible_amount",
    "NCH_BENE_PTA_COINSRNC_LBLTY_AM": "coinsurance_liability_amount",
    "NCH_BENE_BLOOD_DDCTBL_LBLTY_AM": "blood_deductible_liability_amount",
    "CLM_UTLZTN_DAY_CNT": "utilization_days",
    "NCH_BENE_DSCHRG_DT": "discharge_date",
    "CLM_DRG_CD": "drg_code"
}

for old_name, new_name in rename_map.items():
    inpatientdf = inpatientdf.withColumnRenamed(old_name, new_name)


In [13]:
# TypeCasting
from pyspark.sql.functions import col, to_date

inpatientdf = inpatientdf.withColumn("claim_start_date", to_date(col("claim_start_date").cast("string"), "yyyyMMdd")) \
       .withColumn("claim_end_date", to_date(col("claim_end_date").cast("string"), "yyyyMMdd")) \
       .withColumn("admission_date", to_date(col("admission_date").cast("string"), "yyyyMMdd")) \
       .withColumn("discharge_date", to_date(col("discharge_date").cast("string"), "yyyyMMdd"))

In [14]:
# Handle missing values
inpatientdf = inpatientdf.fillna({
    "claim_payment_amount": 0,
    "primary_payer_paid_amount": 0,
    "pass_through_per_diem_amount":0,
    "inpatient_deductible_amount":0,
    "coinsurance_liability_amount":0,
    "blood_deductible_liability_amount":0
})
inpatientdf.show(6)

+----------------+---------------+-------------+----------------+--------------+-----------+--------------------+-------------------------+-----------------------+-----------------------+-------------------+--------------+------------------------+----------------------------+---------------------------+----------------------------+---------------------------------+----------------+--------------+--------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----------+----------+----------+----------+----------+----------+----------+----------+----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+------

### **GOLD** **LAYER**

In [15]:
# printing total cost
from pyspark.sql.functions import col

# Medicare cost.
# We are calculating the total that the insurance company is paying not the customer so we didnot include the other fields here
benecostdf = beneficiarydf.withColumn(
    "TOTAL_COST",
    col("medicare_reimbursement_ip") + col("medicare_reimbursement_op") + col("medicare_reimbursement_car")
)
benecostdf.show()

+----------------+----------+----------+-----------+---------+--------------+----------+-----------+------------------+-------------------+-------------------+--------------------+----------+------------------------+----------------------+------+----+----------+--------+----------------------+------------+----------------------------+-------------+-------------------------+-----------------------------+------------------------+-------------------------+-----------------------------+------------------------+--------------------------+------------------------------+-------------------------+----------+
|      patient_id|birth_date|death_date|gender_code|race_code|esrd_indicator|state_code|county_code|hi_coverage_months|smi_coverage_months|hmo_coverage_months|plan_coverage_months|alzheimers|congestive_heart_failure|chronic_kidney_disease|cancer|copd|depression|diabetes|ischemic_heart_disease|osteoporosis|rheumatoid_or_osteoarthritis|stroke_or_tia|medicare_reimbursement_ip|beneficiary_resp

In [16]:
# 1.Calculating total healthcare cost
# total cost for each provider id and find aggregate for each provider

from pyspark.sql.functions import col, sum, count, countDistinct, avg
provider_gold = (
     inpatientdf
     .withColumn( "total_cost",
                 col("claim_payment_amount") +
                  col("primary_payer_paid_amount") +
                  col("inpatient_deductible_amount") )
     .groupBy("provider_id")
     .agg( sum("total_cost").alias("total_cost"),
    count("claim_id").alias("total_claims"),
    countDistinct("patient_id").alias("total_patients"), avg("total_cost").alias("avg_cost_per_claim")
 ) )

provider_gold.show()

+-----------+----------+------------+--------------+------------------+
|provider_id|total_cost|total_claims|total_patients|avg_cost_per_claim|
+-----------+----------+------------+--------------+------------------+
|     3300PN|   50272.0|           5|             5|           10054.4|
|     01S1YV| 1014264.0|          88|            76|11525.727272727272|
|     1800BG|  742232.0|          63|            46|11781.460317460318|
|     36T0AC|  232996.0|          17|            14| 13705.64705882353|
|     4530WG|  290580.0|          29|            26|           10020.0|
|     1001JT|   24192.0|           3|             3|            8064.0|
|     23T0HR|  240824.0|          27|            21| 8919.407407407407|
|     1002TC|  311612.0|          28|            28|           11129.0|
|     4401WJ|   44260.0|           4|             4|           11065.0|
|     36T0CG|  176920.0|          12|            11|14743.333333333334|
|     10T2SU|    5068.0|           1|             1|            

In [17]:
# 2. Which providers handle most payments

from pyspark.sql.functions import col, sum

provider_df = (
    inpatientdf
    .groupBy("provider_id")
    .agg(
        sum("claim_payment_amount").alias("total_payments")
    )
)

provider_df.orderBy(
    col("total_payments").desc()
).show(10)



+-----------+--------------+
|provider_id|total_payments|
+-----------+--------------+
|     23006G|     7861100.0|
|     1000AH|     6501660.0|
|     2100WC|     4063800.0|
|     3600NG|     4042700.0|
|     2600ZT|     3929800.0|
|     1401RR|     3778400.0|
|     3400XN|     3514600.0|
|     4500DP|     3374700.0|
|     1002DC|     3162300.0|
|     0501MA|     3136800.0|
+-----------+--------------+
only showing top 10 rows


DASHBOARD Bronze -> Silver -> Gold -> Dashboards

In [18]:
# Using tableau for dashboarding

# Step 1: Export data to tableau
provider_gold.toPandas().to_csv(
    "provider_gold.csv",
    index=False
)

provider_gold.show()

+-----------+----------+------------+--------------+------------------+
|provider_id|total_cost|total_claims|total_patients|avg_cost_per_claim|
+-----------+----------+------------+--------------+------------------+
|     3300PN|   50272.0|           5|             5|           10054.4|
|     01S1YV| 1014264.0|          88|            76|11525.727272727272|
|     1800BG|  742232.0|          63|            46|11781.460317460318|
|     36T0AC|  232996.0|          17|            14| 13705.64705882353|
|     4530WG|  290580.0|          29|            26|           10020.0|
|     1001JT|   24192.0|           3|             3|            8064.0|
|     23T0HR|  240824.0|          27|            21| 8919.407407407407|
|     1002TC|  311612.0|          28|            28|           11129.0|
|     4401WJ|   44260.0|           4|             4|           11065.0|
|     36T0CG|  176920.0|          12|            11|14743.333333333334|
|     10T2SU|    5068.0|           1|             1|            